# The posterior sd that was never a posterior sd

A Laplace approximation is the cheapest useful posterior there is: find the mode, take the
curvature there, call it a normal. Every part of that is fine except the middle one, and the
middle one is where the parent repo shipped a bug for two years.

An optimizer like BFGS builds an *approximation* to the inverse Hessian as it goes — a
by-product tuned to make the next step good, not to be correct at the end. It is right there
in `result.hess_inv`, it has the right shape, and taking its diagonal square root gives
numbers that look exactly like posterior standard deviations. They are not. On a hierarchical
model they can be out by a factor of two, in the direction that makes everything look more
certain.

So: everything here happens in unconstrained coordinates (review B13), the mode is found
there, the Hessian is the **real** curvature at the mode — by automatic differentiation where
jax is installed, else by unit-scaled finite differences — and draws are mapped back through
the transforms. If the mode search does not converge, or the Hessian is not positive
definite, you get an `Unverified` with the diagnostics, not a posterior. `nonfinite_draw_frac`
is reported on every result.

In [ ]:
import numpy as np

from axiom.core import D, Data, Gather, Likelihood, ModelSpec, Param, Prior, Unverified, dimensionless, is_failure, value
from axiom.infer import Posterior, find_mode, hessian_at, laplace

from axiom.display import enable

import sys; sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way
from _style import CRITICAL, caption, density, heat, mark_x, scatter_fit

enable();  # every axiom result renders itself from here on

## A hierarchical model — the case that broke the parent

Three units with their own intercepts under a shared hyperprior. Hierarchies are where the
optimizer's approximation goes worst wrong, because the group scale and the group means are
strongly coupled and BFGS's running estimate never has to get that coupling right to
terminate.

In [ ]:
unit = Data(name="unit", dimension=dimensionless())
y = Data(name="y", dimension=D.outcome)
a_mean = Param(name="a_mean", dimension=D.outcome, prior=Prior(family="normal", hyper={"mu": 0.0, "sigma": 5.0}))
a_sd = Param(name="a_sd", dimension=D.outcome, prior=Prior(family="halfnormal", hyper={"sigma": 2.0}))
alpha = Param(name="alpha", dimension=D.outcome, shape=(3,), prior=Prior(family="normal", hyper={"mu": "a_mean", "sigma": "a_sd"}))
sigma = Param(name="sigma", dimension=D.outcome, prior=Prior(family="halfnormal", hyper={"sigma": 2.0}))
model = ModelSpec(name="hier", mean=Gather(source=alpha, index=unit), outcome=y, likelihood=Likelihood(family="normal", scale="sigma"), parameters=(a_mean, a_sd, alpha, sigma))
rng = np.random.default_rng(1)
idx = np.repeat([0, 1, 2], 30)
data = {"unit": idx, "y": np.array([1.0, 2.5, 4.0])[idx] + rng.normal(0, 0.7, 90)}

In [ ]:
from axiom.display import show

mode = find_mode(model, data)
print(mode if is_failure(mode) else (mode.method, mode.converged, {k: np.round(v, 3) for k, v in mode.theta.items()}))
post = laplace(model, data, draws=4000, seed=0)
if isinstance(post, Posterior):
    prov = post.provenance
    print({k: prov[k] for k in ("hessian_pd", "nonfinite_draw_frac", "converged", "min_eigenvalue")})
    print(post.summary("a_sd"), post.draws("alpha").shape)
else:
    show(post)

In [ ]:
truth = np.array([1.0, 2.5, 4.0])
draws = post.flat("alpha")
fig = density(
    {f"alpha[{i}]": draws[:, i] for i in range(3)},
    title="What the approximation actually delivers",
    subtitle="four thousand draws mapped back through the transforms, against the intercepts that generated the data",
    x_title="unit intercept",
)
for t in truth:
    mark_x(fig, float(t), text="")
caption(fig, "Three dotted lines, three posteriors. The draws are normal in unconstrained "
             "space, not in this one — which is why the mapping back is part of the method "
             "rather than a formatting step.")

## The Hessian is real curvature

`hessian_at` evaluates the Hessian of the negative log density at a point — by jax when
installed, else by unit-scaled finite differences. Two independent routes to the same matrix
is how you find out that one of them is an optimizer's souvenir.

In [ ]:
from axiom.core import jax_available, unconstrain

z = unconstrain(model, {"a_mean": 2.5, "a_sd": 1.0, "alpha": np.array([1.0, 2.5, 4.0]), "sigma": 0.7})
H_fd = hessian_at(model, data, z, derivatives="finite_difference")
print(np.round(np.linalg.eigvalsh(H_fd), 2))
if jax_available():
    H_jax = hessian_at(model, data, z, derivatives="jax")
    print("max |H_jax - H_fd| / |H_jax|:", float(np.max(np.abs(H_jax - H_fd)) / np.max(np.abs(H_jax))))

In [ ]:
if jax_available():
    fig = scatter_fit(
        H_jax.ravel(), H_fd.ravel(),
        title="Two routes to the same curvature",
        subtitle="every entry of the Hessian: automatic differentiation against unit-scaled finite differences",
        x_title="jax", y_title="finite difference",
    )
    caption(fig, "This is the check that an optimizer's inverse-Hessian by-product would fail "
                 "loudly — and the reason the fallback route is safe to ship for readers "
                 "without jax installed.")
    fig

In [ ]:
labels = ["a_mean", "a_sd", "alpha[0]", "alpha[1]", "alpha[2]", "sigma"]
cov = np.linalg.inv(H_fd)
sd = np.sqrt(np.diag(cov))
corr = cov / np.outer(sd, sd)
fig = heat(
    corr, labels, labels,
    diverging=True,
    colorbar_title="corr",
    title="Why the coupling has to be right",
    subtitle="posterior correlation implied by the curvature at the mode",
    height=340,
)
caption(fig, "The group scale and the intercepts are correlated by construction. An "
             "approximation that got the diagonal right and this wrong would report the same "
             "standard deviations and the wrong joint uncertainty for every derived quantity.")

## When it cannot be trusted

A product-form mean `a · b` has a saddle at the origin and a ridge everywhere else: only the
product is identified, so the likelihood is flat along a hyperbola. There is no mode to take
curvature at. The mode search restarts, and if it cannot reach a positive-definite mode it
says so.

In [ ]:
a = Param(name="a", dimension=D.outcome, prior=Prior(family="normal", hyper={"mu": 0.0, "sigma": 10.0}))
b = Param(name="b", dimension=dimensionless(), prior=Prior(family="normal", hyper={"mu": 0.0, "sigma": 10.0}))
from axiom.core import Mul

prod = ModelSpec(name="product", mean=Mul(factors=(a, b)), outcome=y, likelihood=Likelihood(family="normal", scale="sigma"), parameters=(a, b, sigma))
pdata = {"y": rng.normal(3.0, 0.5, 40)}
res = laplace(prod, pdata, draws=500, seed=0)
print(type(res).__name__, res.provenance.get("n_restarts") if isinstance(res, Posterior) else res.reason[:120])
flat = laplace(prod, pdata, draws=500, seed=0, allow_unverified=True)
print(type(flat).__name__)

In [ ]:
from axiom.core import log_density

grid = np.linspace(0.4, 6.0, 21)
surface = [[log_density(prod, pdata, unconstrain(prod, {"a": float(av), "b": float(bv), "sigma": 0.5}))
            for av in grid] for bv in grid]
surface = np.array(surface)
surface = np.clip(surface - surface.max(), -60, 0)

fig = heat(
    surface, [f"{g:.1f}" for g in grid], [f"{g:.1f}" for g in grid],
    text_fmt="",
    colorbar_title="log density",
    title="The ridge a normal approximation cannot describe",
    subtitle="log density over (a, b) for mean = a · b — only the product is identified",
    x_title="a", y_title="b",
    height=420,
)
caption(fig, "Every point along the bright curve fits equally well. A method that returned a "
             "mode and a standard deviation here would be reporting the arbitrary spot where "
             "the optimizer stopped — so this one returns Unverified instead, and the "
             "allow_unverified flag makes taking it anyway a deliberate act.")

## What this bought you

A fast posterior you can put weight on, or a refusal with the diagnostics that say why —
never a plausible standard deviation computed from an optimizer's internal bookkeeping.

`nbs/infer/03-convergence.ipynb` does the same job for sampled posteriors, and
`nbs/diagnose/01-sbc-and-coverage.ipynb` is where "put weight on" gets tested rather than
asserted.